## 📝 Problem: Mini Student Exam Score Prediction

### Background: 
You are a data scientist at a small tutoring center. The center wants to know which student factors are most important in predicting exam performance so they can focus their efforts on the right areas.

### Data:

You have 4 student features:

Feature	Description
- X1	Hours studied per day
- X2	Attendance percentage
- X3	Sleep hours per night
- X4	Social media usage hours per day

### The target variable is: 
exam_score → the final exam score (0–100)

The relationship (hidden to you) is:

> exam_score = 5*X1 + 2*X2 + 1.5*X3 - 3*X4 + random noise

### Task: 
- Create a synthetic dataset with 10–20 students (rows) and the 4 features. Add the target column exam_score.
- Implement Best Subset Selection to figure out which combination of features predicts exam scores most accurately.
- Try all subsets of size 1 → 4
- Use Mean Squared Error (MSE) as your evaluation metric

### Output:
- Best feature subset
- Corresponding MSE

Bonus (Optional):
Visualize how the MSE changes as you increase the number of features.
Think about why adding more features doesn’t always reduce error — hint: noise.

### Step 1: Create synthetic datset

In [24]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 100

#create dataframe
df = pd.DataFrame({
    "hours_studied_per_day": np.random.randint( 2, 7, n),
    "attendence_percentage": np.random.randint(30, 87, n),
    "sleep_hours_per_night": np.random.randint(4, 9, n),
    "social_media_usage": np.random.randint(1, 12, n),
    "random_noise" : np.random.randn(n)
    })

#target
df["exam_score"] = ( 5* df["hours_studied_per_day"] 
                     + 2*df["attendence_percentage"] 
                     + 1.5*df["sleep_hours_per_night"] 
                     - 3*df["social_media_usage"] 
                     +  np.random.randn(n) * 5 )
df.head()


,hours_studied_per_day,attendence_percentage,sleep_hours_per_night,social_media_usage,random_noise,exam_score
0,5,57,4,5,-1.991538,122.796736
1,6,36,7,4,0.426026,101.846588
2,4,38,4,8,-0.541285,87.068333
3,6,37,4,8,0.776823,88.022914
4,6,41,5,7,-0.047644,98.950363


### Step 2: Train and test set

In [25]:
from sklearn.model_selection import train_test_split

X = df.drop(columns =  'exam_score')
y = df["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state= 42)

### Step 3: Evaluation function

In [26]:
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

def evaluate( features ):
    model = LinearRegression()

    model.fit(X_train[list(features) ], y_train)

    preds = model.predict(X_test[list(features)])

    mse = mean_squared_error(y_test, preds)

    return mse
    

### Step 4: Best subset selection

In [27]:
from itertools import combinations

features = list(X_train.columns)
best_score = float('inf')
best_subset = None 

for k in range(1 , len(features) + 1):
   
    print(f"checking subsets with {k} features:")

    for subset  in combinations(features , k):
        
        score = evaluate(subset)
        
        print(f" Subset : {subset}, MSE : {score:.2f}")

        #updating best score
        if score < best_score:
            best_score = score
            best_subset = subset
    
print("\n✅ Best Subset of Features:")
print(best_subset)

print("\n✅ Best MSE on Test Set:")
print(best_score)

        

checking subsets with 1 features:
 Subset : ('hours_studied_per_day',), MSE : 1169.45
 Subset : ('attendence_percentage',), MSE : 190.47
 Subset : ('sleep_hours_per_night',), MSE : 1244.38
 Subset : ('social_media_usage',), MSE : 1056.80
 Subset : ('random_noise',), MSE : 1217.77
checking subsets with 2 features:
 Subset : ('hours_studied_per_day', 'attendence_percentage'), MSE : 120.11
 Subset : ('hours_studied_per_day', 'sleep_hours_per_night'), MSE : 1167.93
 Subset : ('hours_studied_per_day', 'social_media_usage'), MSE : 993.88
 Subset : ('hours_studied_per_day', 'random_noise'), MSE : 1133.80
 Subset : ('attendence_percentage', 'sleep_hours_per_night'), MSE : 191.23
 Subset : ('attendence_percentage', 'social_media_usage'), MSE : 66.58
 Subset : ('attendence_percentage', 'random_noise'), MSE : 194.88
 Subset : ('sleep_hours_per_night', 'social_media_usage'), MSE : 1045.17
 Subset : ('sleep_hours_per_night', 'random_noise'), MSE : 1205.68
 Subset : ('social_media_usage', 'random_no